In [11]:
import gradio as gr
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

True

In [12]:
# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
model_name = "gpt-4o-mini"

In [13]:
# System message to guide the assistant's behavior
system_message = "You are a helpful, friendly, and knowledgeable AI assistant."

In [14]:
def chat(message, history):
    """Generate a non-streaming response and print it."""
    # Convert Gradio history format to OpenAI messages format
    messages = [{"role": "system", "content": system_message}]
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": message})
    
    # Get the full response without streaming
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        stream=False,
        temperature=0.7,
        max_tokens=512,
    )
    
    # Print the full response
    full_response = response.choices[0].message.content
    print(full_response)
    return full_response

In [ ]:

"""Interactive terminal chat using the chat() function."""
history = []  # Maintain conversation history in Gradio format

print("\n" + "="*50)
print("GPT-4o Mini Interactive Chat")
print("Type 'quit' or 'exit' to end the conversation")
print("="*50 + "\n")

while True:
    user_input = input("You: ")
    print(f"User: {user_input}")
    print("Assistant: ", end="", flush=True)
    response = chat(user_input, history)
    history.append((user_input, response))

In [ ]:
# Gradio web version with streaming
def chat_response(message, history):
    """Generate a streaming response for Gradio chat interface."""
    # Convert Gradio history format to OpenAI messages format
    messages = [{"role": "system", "content": system_message}]
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": message})
    
    # Stream the response
    stream = client.chat.completions.create(
        model=model_name,
        messages=messages,
        stream=True,
        temperature=0.7,
        max_tokens=512,
    )
    
    # Yield tokens as they come
    response = ""
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            content = chunk.choices[0].delta.content
            response += content
            yield response


# Create Gradio interface
demo = gr.ChatInterface(
    fn=chat_response,
    title="GPT-4o Mini Chatbot",
    description="Chat with GPT-4o Mini using OpenAI API",
    examples=["Hello! How are you?", "What is machine learning?", "Explain quantum computing"],
    theme=gr.themes.Soft(),
)

# Launch the interface
demo.launch(share=False)